# FreeFine Final Geometry — Full 5,677 Metrics: SGR_EPSREC

Attach:
1. the **four completed generation-shard notebook outputs**,
2. `freefine-sample-metadata`,
3. `geobench2d-coarse-img`,
4. `geobench2d-metrics-subset`.

Run on **T4×2 with Internet ON**.

This notebook evaluates all seven official metrics on:
- all 5,677 edits
- move 1,439
- rotate 1,603
- resize 2,635
- exact-affine severe resize 759
- exact-affine non-severe resize 1,876

It is resume-safe: if it ends `PARTIAL METRICS`, save the version, attach that saved output, and rerun this same notebook.


In [1]:

# ===== 1. Clone exact FreeFine source + clock =====
import os,time,subprocess,glob,json,shutil,csv,hashlib,math,socket,re
from collections import defaultdict,Counter

NB_START=time.time()
COMMIT="4c9fdb971572b32edbeac13464659274c28decbb"
subprocess.run(
    f"mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine && "
    f"git clone -q https://github.com/CIawevy/FreeFine.git && "
    f"cd FreeFine && git checkout -q {COMMIT}",
    shell=True,check=True
)
print("✓ cloned + pinned FreeFine",COMMIT[:14])


✓ cloned + pinned FreeFine 4c9fdb971572b3


In [2]:

%%bash
set -e
pip install -q --root-user-action=ignore uv
uv python install 3.10.13
V=/kaggle/temp/metric_env
PY=$V/bin/python
REPO=/kaggle/temp/FreeFine
rm -rf "$V"
uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python "$PY" "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/m.txt
uv pip install --python "$PY" -r /tmp/m.txt
uv pip install --python "$PY" "setuptools<70"
uv pip install --python "$PY" --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python "$PY" "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $V -path '*/site-packages/clip' -o -path '*open_clip' -type d); do
  cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null || true
done
echo "✓ metric_env OK"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 87.9 MB/s eta 0:00:00
✓ metric_env OK


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Installed Python 3.10.13 in 1.44s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/metric_env
Activate with: source /kaggle/temp/metric_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/metric_env
Resolved 27 packages in 1.07s
 Downloaded networkx
 Downloaded torchaudio
 Downloaded pillow
 Downloaded sympy
 Downloaded torchvision
 Downloaded numpy
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded nvidia-nvjitlink-cu12
 Downloaded nvidia-curand-cu12
 Downloaded triton
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-cufft-cu12
 Downloaded nvidia-cusolver-cu12
 Downloaded nvidia-cusparse-cu12
 Downloaded nvidia-cusparselt-cu12
 Downloaded nvidia-cublas-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded torch
Prepared 27 packages in 27.21s
Installed 27 packages in 341ms
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1

In [3]:

# ===== Deterministic official metric patches =====
import pathlib,re,py_compile,os
mr=pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
mp=mr/"main.py"
s=mp.read_text()
s=s.replace("args.3d","getattr(args,'3d')")
anchor="    args = parser.parse_args()\n"
assert anchor in s
seed_patch=anchor+"""    import random as _random, numpy as _np
    try:
        import torch as _torch
    except Exception:
        _torch=None
    _metric_seed=int(os.environ.get('FF_METRIC_SEED','42'))
    _random.seed(_metric_seed); _np.random.seed(_metric_seed)
    if _torch is not None:
        _torch.manual_seed(_metric_seed)
        if _torch.cuda.is_available(): _torch.cuda.manual_seed_all(_metric_seed)
"""
s=s.replace(anchor,seed_patch,1)
mp.write_text(s)

for f in [mr/"MD"/"mean_distance.py",mr/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace(
        "stabilityai/stable-diffusion-2-1",
        "sd2-community/stable-diffusion-2-1"
    ))
md=mr/"MD"/"mean_distance.py"
s=md.read_text()
needle="all_dist = []"
assert needle in s
s=s.replace(
    needle,
    needle+"\n    import torch as _st, os as _os; "
           "_seed=int(_os.environ.get('FF_MD_SEED','42')); "
           "_st.manual_seed(_seed); _st.cuda.manual_seed_all(_seed)",
    1
)
md.write_text(s)
for f in [mr/"main.py",mr/"MD"/"mean_distance.py",mr/"MD"/"dift_sd.py"]:
    py_compile.compile(str(f),doraise=True)
print("✓ deterministic metric patches installed")


✓ deterministic metric patches installed


In [4]:

# ===== Full final model assembly from four completed generation shards =====
MODEL_DIR="SGR_EPSREC"
RESULT_SLUG="SGR_EPSREC"
EXPECTED_SHA="17d971102daca232921d33de91adac58ef0ddbea9c9f814e219148486c9de802"

import os,glob,json,csv,shutil,math,hashlib,re
from collections import defaultdict,Counter

GEO="/kaggle/temp/GeoBenchMeta"
os.makedirs(f"{GEO}/Geo-Bench-2D",exist_ok=True)

def one(pattern,desc):
    xs=glob.glob(pattern,recursive=True)
    if not xs: raise FileNotFoundError(f"Missing {desc}: {pattern}")
    return sorted(xs,key=lambda x:(len(x),x))[0]

CACHE=next(c for c in glob.glob("/kaggle/input/**/Geo-Bench-2D",recursive=True)
           if os.path.isdir(f"{c}/source_img"))
COARSE=one("/kaggle/input/**/coarse_img/*/*/*.png","coarse_img").split("/coarse_img/")[0]+"/coarse_img"
ANNP=one("/kaggle/input/**/annotation_2d.json","annotation_2d.json")
META=one("/kaggle/input/**/sample_metadata.csv","sample_metadata.csv")
for nm in ["source_img","source_mask","target_mask","source_img_full_v2"]:
    d=f"{GEO}/Geo-Bench-2D/{nm}"
    if os.path.lexists(d):
        os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
    os.symlink(f"{CACHE}/{nm}",d)
d=f"{GEO}/Geo-Bench-2D/coarse_img"
if os.path.lexists(d):
    os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
os.symlink(COARSE,d)
shutil.copy(ANNP,f"{GEO}/annotation_2d.json")
ann=json.load(open(f"{GEO}/annotation_2d.json"))

meta=[]
for r in csv.DictReader(open(META)):
    if r["edit_type"] not in ("move","rotate","resize"): continue
    d,i,e=str(r["da_n"]),str(r["ins_id"]),str(r["case_id"])
    ep=ann[d]["instances"][i][e]["edit_param"]
    sx,sy=float(ep[6]),float(ep[7])
    scale=math.sqrt(sx*sy)
    rr=dict(r)
    rr.update({"da_n":d,"ins_id":i,"case_id":e,"scale_exact":scale,
               "affine_severe":(r["edit_type"]=="resize" and (scale>=1.5 or scale<=0.6))})
    meta.append(rr)

assert len(meta)==5677
assert Counter(r["edit_type"] for r in meta)=={"resize":2635,"rotate":1603,"move":1439}
assert sum(r["affine_severe"] for r in meta)==759

# Verify exact affine fingerprint.
lines=[]
for r in sorted(meta,key=lambda z:(z["da_n"],z["ins_id"],z["case_id"])):
    vals=[float(x) for x in ann[r["da_n"]]["instances"][r["ins_id"]][r["case_id"]]["edit_param"]]
    lines.append(f"{r['da_n']}|{r['ins_id']}|{r['case_id']}|{r['edit_type']}|"+
                 "|".join(f"{x:.12g}" for x in vals))
sha=hashlib.sha256("\n".join(lines).encode()).hexdigest()
assert sha==EXPECTED_SHA,(sha,EXPECTED_SHA)

# Same deterministic shard assignment used by generation.
def category(r):
    if r["edit_type"]=="move": return "move"
    if r["edit_type"]=="rotate": return "rotate"
    return "resize_severe" if r["affine_severe"] else "resize_nonsevere"
offset={"move":0,"rotate":1,"resize_nonsevere":0,"resize_severe":2}
bycat=defaultdict(list)
for r in meta: bycat[category(r)].append(r)
row_shard={}
for cat,rows in bycat.items():
    rows=sorted(rows,key=lambda z:(z["da_n"],z["ins_id"],z["case_id"]))
    for j,r in enumerate(rows):
        row_shard[(r["da_n"],r["ins_id"],r["case_id"])]=(j+offset[cat])%4

# Find the most complete attached output for each shard.
def pc(p): return len(glob.glob(p+"/**/*.png",recursive=True)) if os.path.isdir(p) else 0
roots={}
for s in range(4):
    cands=glob.glob(f"/kaggle/input/**/final_geometry_full/shard_{s}/{MODEL_DIR}",recursive=True)
    if not cands:
        raise FileNotFoundError(f"Missing completed shard {s} for {MODEL_DIR}")
    root=max(cands,key=pc)
    roots[s]=root
    print("shard",s,pc(root),root)

# Assemble exact 5,677 model outputs by expected sample->shard mapping.
EVALROOT=f"{GEO}/gen_eval/{RESULT_SLUG}"
if os.path.exists(EVALROOT): shutil.rmtree(EVALROOT)
os.makedirs(EVALROOT,exist_ok=True)

missing=[]
for r in meta:
    k=(r["da_n"],r["ins_id"],r["case_id"])
    s=row_shard[k]
    src=f"{roots[s]}/{r['da_n']}/{r['ins_id']}/{r['case_id']}.png"
    dst=f"{EVALROOT}/{r['da_n']}/{r['ins_id']}/{r['case_id']}.png"
    if not os.path.exists(src):
        missing.append((k,s,src)); continue
    os.makedirs(os.path.dirname(dst),exist_ok=True)
    os.symlink(os.path.realpath(src),dst)

assert not missing,missing[:20]
assert len(glob.glob(EVALROOT+"/**/*.png",recursive=True))==5677
print("✓ assembled",RESULT_SLUG,"5677/5677")

def keys(pred):
    return [(r["da_n"],r["ins_id"],r["case_id"]) for r in meta if pred(r)]
GROUPS={
    "all_5677":keys(lambda r:True),
    "move_1439":keys(lambda r:r["edit_type"]=="move"),
    "rotate_1603":keys(lambda r:r["edit_type"]=="rotate"),
    "resize_2635":keys(lambda r:r["edit_type"]=="resize"),
    "resize_severe_759":keys(lambda r:r["edit_type"]=="resize" and r["affine_severe"]),
    "resize_nonsevere_1876":keys(lambda r:r["edit_type"]=="resize" and not r["affine_severe"]),
}
print("groups",{k:len(v) for k,v in GROUPS.items()})


shard 0 1419 /kaggle/input/notebooks/georgiostzamouranis/01-full5677-generation-shard0of4-t4x2-v1/final_geometry_full/shard_0/SGR_EPSREC


shard 1 1419 /kaggle/input/notebooks/giorgostzam/01-full5677-generation-shard1of4-t4x2-v1/final_geometry_full/shard_1/SGR_EPSREC


shard 2 1420 /kaggle/input/notebooks/papadonikolas/01-full5677-generation-shard2of4-t4x2-v1/final_geometry_full/shard_2/SGR_EPSREC


shard 3 1419 /kaggle/input/notebooks/gtz19800/01-full5677-generation-shard3of4-t4x2-v1/final_geometry_full/shard_3/SGR_EPSREC


✓ assembled SGR_EPSREC 5677/5677
groups {'all_5677': 5677, 'move_1439': 1439, 'rotate_1603': 1603, 'resize_2635': 2635, 'resize_severe_759': 759, 'resize_nonsevere_1876': 1876}


In [5]:

# ===== Run all seven official metrics on every final reporting group =====
import os,json,re,subprocess,time,threading,queue,pandas as pd,glob,shutil

MET="/kaggle/temp/FreeFine/evaluation/metrics"
PY="/kaggle/temp/metric_env/bin/python"
OUT_JSON=f"/kaggle/working/final_metric_results_{RESULT_SLUG}.json"
OUT_CSV=f"/kaggle/working/final_metric_results_{RESULT_SLUG}.csv"
LOGDIR=f"/kaggle/working/metric_logs_{RESULT_SLUG}"
os.makedirs(LOGDIR,exist_ok=True)
DEADLINE=NB_START+11.55*3600

# Resume from a previous saved evaluation version if attached.
results={}
prev=glob.glob(f"/kaggle/input/**/final_metric_results_{RESULT_SLUG}.json",recursive=True)
if prev:
    try:
        results=json.load(open(max(prev,key=os.path.getsize)))
        print("resumed metric JSON with groups",list(results))
    except Exception:
        results={}

def manifest(gname,ids):
    o={}
    used=0
    for d,i,e in ids:
        gp=f"{EVALROOT}/{d}/{i}/{e}.png"
        if not os.path.exists(gp): continue
        lf=dict(ann[d]["instances"][i][e])
        lf["gen_img_path"]=f"gen_eval/{RESULT_SLUG}/{d}/{i}/{e}.png"
        o.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
        used+=1
    p=f"{GEO}/m_{RESULT_SLUG}_{gname}.json"
    json.dump(o,open(p,"w"))
    return p,used

def complete(v):
    return all(k in v for k in ["SUBC","BGC","WRAP_E","MD","FID","FID_DINO","FID_KD"])

def run_one(gpu,gname):
    man,n=manifest(gname,GROUPS[gname])
    assert n==len(GROUPS[gname]),(gname,n,len(GROUPS[gname]))
    env=os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES":str(gpu),
        "MPLBACKEND":"Agg",
        "HF_HOME":"/kaggle/temp/hf",
        "TORCH_HOME":"/kaggle/temp/torch",
        "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
        "FF_MD_SEED":"42",
        "FF_METRIC_SEED":"42",
        "TOKENIZERS_PARALLELISM":"false",
    })
    logp=f"{LOGDIR}/{gname}.log"
    p=subprocess.run(
        [PY,"main.py","--path",man,"--use_relative_path","--base_dir",GEO,
         "--fid_path",f"{GEO}/Geo-Bench-2D/source_img_full_v2",
         "--task","100111111","--level","0"],
        cwd=MET,env=env,capture_output=True,text=True
    )
    txt=p.stdout+p.stderr
    open(logp,"w").write(txt)
    vals={"n":n}
    for k in ["FID_DINO","FID_KD","FID","SUBC","BGC","WRAP_E","MD"]:
        h=re.findall(rf"(?:^|\s){k}:\s*([-\d.eE]+)",txt)
        if h: vals[k]=round(float(h[-1]),4)
    if p.returncode!=0:
        vals["_rc"]=p.returncode
        vals["_tail"]=txt[-2000:]
    return vals

order=["all_5677","resize_2635","move_1439","rotate_1603","resize_severe_759","resize_nonsevere_1876"]
jobs=queue.Queue()
for g in order:
    if not complete(results.get(g,{})):
        jobs.put(g)

lock=threading.Lock()
def save():
    with lock:
        json.dump(results,open(OUT_JSON,"w"),indent=2)
        rows=[{"model":RESULT_SLUG,"group":g,**v} for g,v in results.items()]
        pd.DataFrame(rows).to_csv(OUT_CSV,index=False)

def worker(gpu):
    while time.time()<DEADLINE-120:
        try: g=jobs.get_nowait()
        except queue.Empty:return
        try:
            print(f"[GPU{gpu}] {g} start",flush=True)
            v=run_one(gpu,g)
            with lock: results[g]=v
            save()
            print(f"[GPU{gpu}] {g} -> {v}",flush=True)
        finally:
            jobs.task_done()

ts=[threading.Thread(target=worker,args=(g,),daemon=True) for g in [0,1]]
[t.start() for t in ts]
[t.join() for t in ts]
save()

missing=[g for g in order if not complete(results.get(g,{}))]
print("MISSING",missing)
if not missing:
    print("✓ FULL FINAL METRICS COMPLETE",RESULT_SLUG)
else:
    print("PARTIAL METRICS — Save Version, attach this output, and rerun the same evaluation notebook.")
print(OUT_JSON)
print(OUT_CSV)


[GPU0] all_5677 start


[GPU1] resize_2635 start


[GPU1] resize_2635 -> {'n': 2635, 'FID_DINO': 532.2316, 'FID_KD': 0.1381, 'FID': 40.6894, 'SUBC': 0.8942, 'BGC': 0.967, 'WRAP_E': 0.03, 'MD': 9.5589}


[GPU1] move_1439 start


[GPU1] move_1439 -> {'n': 1439, 'FID_DINO': 581.6613, 'FID_KD': 0.1248, 'FID': 47.2753, 'SUBC': 0.9665, 'BGC': 0.9672, 'WRAP_E': 0.0121, 'MD': 3.9897}


[GPU1] rotate_1603 start


[GPU0] all_5677 -> {'n': 5677, 'FID_DINO': 487.358, 'FID_KD': 0.1417, 'FID': 34.7793, 'SUBC': 0.9144, 'BGC': 0.9671, 'WRAP_E': 0.0292, 'MD': 8.1779}
